# 09 — XGBoost Forecasting Candidate v2

**Gold Nexus Alpha — professor-style XGBoost revision**

This notebook revises XGBoost so it is evaluated in the same fair daily-forecasting style as the Naive, Regression, and revised ARIMA/SARIMAX notebooks.

Main corrections:

1. Keep Dataset B and the locked validation/test periods.
2. Exclude `high_yield` from the main model.
3. Use lagged macro variables to reduce leakage risk.
4. Test different target designs because tree models can struggle to extrapolate price levels.
5. Compare fixed one-step validation forecasts first.
6. Add rolling refit forecasts for the top candidates.
7. Select the final XGBoost candidate by rolling validation RMSE.
8. Export feature importance and optional SHAP summary.
9. Export JSON artifacts for the frontend and later model comparison notebook.

Important interpretation:

- XGBoost can underforecast if it predicts the raw gold price level beyond the range seen in training.
- Predicting daily change, log return, or log price can be more stable.
- Rolling refit helps the model adapt as new observations become available.

In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the clean Colab → GitHub artifact workflow used across Gold Nexus Alpha.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

print("✅ Repo ready:", PROJECT_DIR)
print("✅ Branch:", BRANCH)
print("✅ UTC time:", datetime.now(timezone.utc).isoformat())

In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load XGBoost, scikit-learn, and plotting tools.
# - Use Colab GPU when available, with CPU fallback.
# ======================================================================================

import sys
import json
import math
import glob
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import sklearn
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    import sklearn

try:
    import xgboost as xgb
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    import xgboost as xgb

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 180)

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("SHAP available:", SHAP_AVAILABLE)

In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Dataset B: data/aligned/model_ready_multivariate.csv
# - Build leakage-safe lagged macro features.
# - Lock the official train / validation / test dates.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
INTERPRETABILITY_DIR = ARTIFACTS_DIR / "interpretability"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, INTERPRETABILITY_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

CORE_MULTIVARIATE_START = "2006-01-02"
CORE_MULTIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "2006-01-02"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

TARGET_COL = "gold_price"

EXCLUDED_FROM_MAIN_MODEL = ["high_yield"]

CORE_MACRO_FACTORS = [
    "real_yield",
    "nominal_yield",
    "tips_curve",
    "fed_bs",
    "m2_supply",
    "inflation",
    "usd_index",
    "eur_usd",
    "jpy_usd",
    "vix_index",
    "fin_stress",
    "gpr_index",
    "policy_unc",
    "oil_wti",
    "ppi_index",
    "gld_tonnes",
    "unrate",
    "ind_prod",
    "cap_util",
]

GOLD_HISTORY_FEATURES = [
    "gold_lag_1",
    "gold_lag_5",
    "gold_lag_20",
    "gold_return_1",
    "gold_return_5",
    "gold_ma_5",
    "gold_ma_20",
    "gold_ma_60",
    "gold_volatility_20",
]

TARGET_MODES = [
    "price_level",
    "dollar_change",
    "log_price",
    "log_return",
]

MAX_FIXED_CANDIDATES_TO_ROLL = 3
ROLLING_REFIT_EVERY = 20
USE_GPU_IF_AVAILABLE = True
RANDOM_SEED = 42

candidate_inputs = [
    ALIGNED_DIR / "model_ready_multivariate.csv",
    ALIGNED_DIR / "weekday_clean_matrix.csv",
    PROJECT_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
]

candidate_inputs += sorted(Path("/content").glob("*model*ready*multivariate*.csv"))
candidate_inputs += sorted(Path("/content").glob("*Gold*Matrix*.csv"))
candidate_inputs += sorted(Path("/content").glob("*gold*matrix*.csv"))

INPUT_PATH = None
for path in candidate_inputs:
    if path.exists():
        INPUT_PATH = path
        break

if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find model_ready_multivariate.csv, weekday_clean_matrix.csv, or an uploaded Gold_Matrix CSV. "
        "Run Notebook 03 first or upload the current matrix CSV."
    )

print("✅ Input detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")
if TARGET_COL not in raw_df.columns:
    raise ValueError(f"Input file must contain target column '{TARGET_COL}'.")

raw_df["date"] = pd.to_datetime(raw_df["date"], errors="coerce")
raw_df = raw_df.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

raw_df["weekday"] = raw_df["date"].dt.dayofweek
weekend_rows_detected = int(raw_df["weekday"].isin([5, 6]).sum())
if weekend_rows_detected > 0:
    print(f"⚠️ Weekend rows detected in fallback input: {weekend_rows_detected}. Removing Saturday/Sunday rows.")
    raw_df = raw_df[~raw_df["weekday"].isin([5, 6])].copy()
raw_df = raw_df.drop(columns=["weekday"], errors="ignore")

# Enforce Dataset B and remove high_yield from the main model.
available_macro = [c for c in CORE_MACRO_FACTORS if c in raw_df.columns and c not in EXCLUDED_FROM_MAIN_MODEL]
missing_macro = [c for c in CORE_MACRO_FACTORS if c not in raw_df.columns]
if missing_macro:
    print("⚠️ Missing planned macro factors. Proceeding with available factors only:", missing_macro)

needed_cols = ["date", TARGET_COL] + available_macro
for col in GOLD_HISTORY_FEATURES:
    if col in raw_df.columns and col not in needed_cols:
        needed_cols.append(col)

model_df = raw_df[
    (raw_df["date"] >= pd.Timestamp(CORE_MULTIVARIATE_START)) &
    (raw_df["date"] <= pd.Timestamp(CORE_MULTIVARIATE_END))
][needed_cols].copy()

for col in model_df.columns:
    if col != "date":
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# If Notebook 03 features are unavailable, rebuild the basic gold history features safely here.
if "gold_lag_1" not in model_df.columns:
    model_df["gold_lag_1"] = model_df[TARGET_COL].shift(1)
if "gold_lag_5" not in model_df.columns:
    model_df["gold_lag_5"] = model_df[TARGET_COL].shift(5)
if "gold_lag_20" not in model_df.columns:
    model_df["gold_lag_20"] = model_df[TARGET_COL].shift(20)
if "gold_return_1" not in model_df.columns:
    model_df["gold_return_1"] = model_df[TARGET_COL].pct_change(1)
if "gold_return_5" not in model_df.columns:
    model_df["gold_return_5"] = model_df[TARGET_COL].pct_change(5)
if "gold_ma_5" not in model_df.columns:
    model_df["gold_ma_5"] = model_df[TARGET_COL].shift(1).rolling(5).mean()
if "gold_ma_20" not in model_df.columns:
    model_df["gold_ma_20"] = model_df[TARGET_COL].shift(1).rolling(20).mean()
if "gold_ma_60" not in model_df.columns:
    model_df["gold_ma_60"] = model_df[TARGET_COL].shift(1).rolling(60).mean()
if "gold_volatility_20" not in model_df.columns:
    model_df["gold_volatility_20"] = model_df[TARGET_COL].pct_change().shift(1).rolling(20).std()

# Lag macro factors by one row/day to reduce same-day information dependency.
for col in available_macro:
    model_df[f"{col}_lag1"] = model_df[col].shift(1)

model_df["trend_idx"] = np.arange(len(model_df), dtype=float)
model_df["trend_sq"] = model_df["trend_idx"] ** 2
model_df["month"] = model_df["date"].dt.month
model_df["month_sin"] = np.sin(2 * np.pi * model_df["month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["month"] / 12)

macro_lag_cols = [f"{c}_lag1" for c in available_macro]
gold_feature_cols = [c for c in GOLD_HISTORY_FEATURES if c in model_df.columns]
deterministic_cols = ["trend_idx", "trend_sq", "month_sin", "month_cos"]

FEATURE_COLS = gold_feature_cols + macro_lag_cols + deterministic_cols
FEATURE_COLS = [c for c in FEATURE_COLS if c in model_df.columns and c not in [TARGET_COL, "high_yield"]]

# Target transforms require positive gold price and gold_lag_1.
model_df = model_df[model_df[TARGET_COL] > 0].copy()
model_df = model_df.dropna(subset=[TARGET_COL, "gold_lag_1"] + FEATURE_COLS).sort_values("date").reset_index(drop=True)

train_df = model_df[(model_df["date"] >= TRAIN_START) & (model_df["date"] <= TRAIN_END)].copy()
validation_df = model_df[(model_df["date"] >= VALIDATION_START) & (model_df["date"] <= VALIDATION_END)].copy()
test_df = model_df[(model_df["date"] >= TEST_START) & (model_df["date"] <= TEST_END)].copy()

if train_df.empty or validation_df.empty or test_df.empty:
    raise ValueError("One or more locked split windows are empty. Check Notebook 03 outputs and dates.")

print("✅ Dataset B ready for XGBoost")
print("Rows:", len(model_df))
print("Train rows:", len(train_df), train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Validation rows:", len(validation_df), validation_df["date"].min().date(), "to", validation_df["date"].max().date())
print("Test rows:", len(test_df), test_df["date"].min().date(), "to", test_df["date"].max().date())
print("Feature count:", len(FEATURE_COLS))
print("Excluded from main XGBoost:", EXCLUDED_FROM_MAIN_MODEL)
display(model_df[["date", TARGET_COL, "gold_lag_1"] + FEATURE_COLS[:8]].head())
display(model_df[["date", TARGET_COL, "gold_lag_1"] + FEATURE_COLS[:8]].tail())

In [ ]:
# ======================================================================================
# CELL 4 — MAIN XGBOOST MODELING LOGIC
# ======================================================================================
# Purpose:
# - Compare target designs.
# - Add rolling refit evaluation.
# - Use GPU if available, with CPU fallback.
# ======================================================================================


def json_safe(value):
    """Convert pandas/numpy objects into JSON-safe Python values."""
    import pandas as pd
    import numpy as np
    import math
    if value is None:
        return None
    if isinstance(value, (pd.Timestamp,)):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    if isinstance(value, (np.ndarray, list, tuple)):
        return [json_safe(x) for x in value]
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    return value

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    clean_payload = json_safe(payload)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean_payload, f, indent=2)
    print(f"✅ Wrote {path}")

def metrics(actual, prediction):
    actual = pd.Series(actual).astype(float)
    prediction = pd.Series(prediction).astype(float)
    aligned = pd.concat([actual.rename("actual"), prediction.rename("prediction")], axis=1).dropna()
    if aligned.empty:
        return {"n": 0, "mae": None, "mse": None, "rmse": None, "mape": None}
    err = aligned["actual"] - aligned["prediction"]
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    denom = aligned["actual"].replace(0, np.nan)
    mape = float((np.abs(err / denom).dropna().mean()) * 100)
    return {"n": int(len(aligned)), "mae": mae, "mse": mse, "rmse": rmse, "mape": mape}

def path_records(df, max_rows=None):
    out = df.copy()
    if max_rows is not None and len(out) > max_rows:
        out = out.tail(max_rows).copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
    return out.replace({np.nan: None}).to_dict(orient="records")

def add_split_lines(ax, validation_start, test_start):
    ax.axvline(pd.Timestamp(validation_start), linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(pd.Timestamp(test_start), linestyle="--", linewidth=1, alpha=0.7)


RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

# 1) Plot target first.
fig, ax = plt.subplots(figsize=(14, 5))
plot_df = model_df.set_index("date")
plot_df[TARGET_COL].plot(ax=ax)
ax.set_title("Gold Price Time Series — Dataset B for XGBoost")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
add_split_lines(ax, VALIDATION_START, TEST_START)
plt.show()

def target_for_mode(df, mode):
    if mode == "price_level":
        return df[TARGET_COL].astype(float)
    if mode == "dollar_change":
        return (df[TARGET_COL] - df["gold_lag_1"]).astype(float)
    if mode == "log_price":
        return np.log(df[TARGET_COL].astype(float))
    if mode == "log_return":
        return np.log(df[TARGET_COL].astype(float) / df["gold_lag_1"].astype(float))
    raise ValueError(f"Unknown target mode: {mode}")

def reconstruct_price_from_prediction(df, raw_prediction, mode):
    raw_prediction = np.asarray(raw_prediction, dtype=float)
    if mode == "price_level":
        return raw_prediction
    if mode == "dollar_change":
        return df["gold_lag_1"].values.astype(float) + raw_prediction
    if mode == "log_price":
        return np.exp(raw_prediction)
    if mode == "log_return":
        return df["gold_lag_1"].values.astype(float) * np.exp(raw_prediction)
    raise ValueError(f"Unknown target mode: {mode}")

def clean_xy(df, feature_cols, mode):
    temp = df.copy()
    temp["_target_mode_value"] = target_for_mode(temp, mode)
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols + ["_target_mode_value", TARGET_COL, "gold_lag_1"])
    X = temp[feature_cols].astype(float)
    y = temp["_target_mode_value"].astype(float)
    return temp, X, y

def make_xgb_params(gpu_attempt=True):
    params = {
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.035,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 3,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "objective": "reg:squarederror",
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
    }
    if gpu_attempt and USE_GPU_IF_AVAILABLE:
        # XGBoost 2.x uses device='cuda' with tree_method='hist'.
        params.update({"tree_method": "hist", "device": "cuda"})
    else:
        params.update({"tree_method": "hist"})
    return params

def fit_xgb_model(X_train, y_train):
    attempts = [
        make_xgb_params(gpu_attempt=True),
        {**make_xgb_params(gpu_attempt=False), "tree_method": "gpu_hist"},
        make_xgb_params(gpu_attempt=False),
    ]

    last_error = None
    for params in attempts:
        try:
            model = XGBRegressor(**params)
            model.fit(X_train, y_train)
            return model, params, None
        except Exception as e:
            last_error = str(e)
            continue

    raise RuntimeError(f"All XGBoost fit attempts failed. Last error: {last_error}")

def fixed_forecast_for_mode(train_source, eval_source, feature_cols, mode):
    train_clean, X_train, y_train = clean_xy(train_source, feature_cols, mode)
    eval_clean, X_eval, y_eval_mode = clean_xy(eval_source, feature_cols, mode)

    model, params, fit_error = fit_xgb_model(X_train, y_train)
    raw_pred = model.predict(X_eval)
    pred_price = reconstruct_price_from_prediction(eval_clean, raw_pred, mode)

    path = eval_clean[["date", TARGET_COL, "gold_lag_1"]].copy()
    path["prediction"] = pred_price
    path["raw_prediction"] = raw_pred
    path["target_mode"] = mode
    path["actual"] = path[TARGET_COL]
    path["residual"] = path["actual"] - path["prediction"]
    path["abs_error"] = path["residual"].abs()

    return model, params, path

def rolling_refit_forecast(train_history_source, eval_source, feature_cols, mode, refit_every=20):
    history = train_history_source.copy().sort_values("date").reset_index(drop=True)
    eval_sorted = eval_source.copy().sort_values("date").reset_index(drop=True)

    model = None
    params_used = None
    rows = []
    refit_count = 0

    for i, (_, row) in enumerate(eval_sorted.iterrows()):
        if model is None or (refit_every and i % refit_every == 0):
            hist_clean, X_hist, y_hist = clean_xy(history, feature_cols, mode)
            model, params_used, _ = fit_xgb_model(X_hist, y_hist)
            refit_count += 1

        row_df = pd.DataFrame([row])
        row_clean, X_row, _ = clean_xy(row_df, feature_cols, mode)

        if row_clean.empty:
            continue

        raw_pred = model.predict(X_row)
        pred_price = reconstruct_price_from_prediction(row_clean, raw_pred, mode)[0]
        actual = float(row_clean[TARGET_COL].iloc[0])

        rows.append({
            "date": row_clean["date"].iloc[0],
            "actual": actual,
            "prediction": float(pred_price),
            "raw_prediction": float(raw_pred[0]),
            "residual": actual - float(pred_price),
            "abs_error": abs(actual - float(pred_price)),
            "target_mode": mode,
            "refit_count_so_far": refit_count,
        })

        # Append actual observed row to history for future refits.
        history = pd.concat([history, row_df], axis=0, ignore_index=True)

    return pd.DataFrame(rows), refit_count, params_used

# 2) Fixed one-step validation across target designs.
fixed_validation_rows = []
fixed_validation_objects = {}

for mode in TARGET_MODES:
    try:
        model, params, path = fixed_forecast_for_mode(train_df, validation_df, FEATURE_COLS, mode)
        m = metrics(path["actual"], path["prediction"])
        fixed_validation_rows.append({
            "candidate_id": f"xgboost_{mode}",
            "model_name": "XGBoost",
            "target_mode": mode,
            "forecast_mode": "fixed_model_one_step_validation",
            **m,
            "fit_params": params,
            "fit_error": None,
        })
        fixed_validation_objects[f"xgboost_{mode}"] = {
            "model": model,
            "params": params,
            "path": path,
            "target_mode": mode,
        }
    except Exception as e:
        fixed_validation_rows.append({
            "candidate_id": f"xgboost_{mode}",
            "model_name": "XGBoost",
            "target_mode": mode,
            "forecast_mode": "fixed_model_one_step_validation",
            "n": 0,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "fit_params": {},
            "fit_error": str(e),
        })

fixed_validation_table = pd.DataFrame(fixed_validation_rows).sort_values("rmse", na_position="last").reset_index(drop=True)
print("\nFixed one-step XGBoost validation leaderboard")
display(fixed_validation_table)

valid_fixed_candidates = fixed_validation_table.dropna(subset=["rmse"]).head(MAX_FIXED_CANDIDATES_TO_ROLL).copy()
if valid_fixed_candidates.empty:
    raise RuntimeError("No XGBoost fixed validation candidates fitted successfully.")

# 3) Rolling refit validation for top candidates.
rolling_validation_rows = []
rolling_validation_paths = {}

for _, row in valid_fixed_candidates.iterrows():
    mode = row["target_mode"]
    candidate_id = row["candidate_id"]
    try:
        path, refit_count, params_used = rolling_refit_forecast(
            train_df,
            validation_df,
            FEATURE_COLS,
            mode,
            refit_every=ROLLING_REFIT_EVERY
        )
        m = metrics(path["actual"], path["prediction"])
        rolling_validation_rows.append({
            "candidate_id": candidate_id,
            "model_name": "XGBoost",
            "target_mode": mode,
            "forecast_mode": "rolling_refit_one_step_validation",
            **m,
            "refit_every": ROLLING_REFIT_EVERY,
            "refit_count": int(refit_count),
            "fit_params": params_used,
            "fit_error": None,
        })
        rolling_validation_paths[candidate_id] = path
    except Exception as e:
        rolling_validation_rows.append({
            "candidate_id": candidate_id,
            "model_name": "XGBoost",
            "target_mode": mode,
            "forecast_mode": "rolling_refit_one_step_validation",
            "n": 0,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "refit_every": ROLLING_REFIT_EVERY,
            "refit_count": None,
            "fit_params": {},
            "fit_error": str(e),
        })

rolling_validation_table = pd.DataFrame(rolling_validation_rows).sort_values("rmse", na_position="last").reset_index(drop=True)
print("\nRolling refit XGBoost validation leaderboard")
display(rolling_validation_table)

if rolling_validation_table.dropna(subset=["rmse"]).empty:
    raise RuntimeError("No XGBoost rolling validation candidates completed successfully.")

selected_row = rolling_validation_table.dropna(subset=["rmse"]).iloc[0].to_dict()
SELECTED_CANDIDATE_ID = selected_row["candidate_id"]
SELECTED_TARGET_MODE = selected_row["target_mode"]

print("\n✅ Selected XGBoost candidate by rolling validation RMSE:", SELECTED_CANDIDATE_ID)
display(pd.DataFrame([selected_row]))

# 4) Final test evaluation.
train_validation_df = pd.concat([train_df, validation_df], axis=0).sort_values("date").reset_index(drop=True)

# Static/fixed test diagnostic.
selected_fixed_model, selected_fixed_params, fixed_test_path = fixed_forecast_for_mode(
    train_validation_df,
    test_df,
    FEATURE_COLS,
    SELECTED_TARGET_MODE
)
fixed_test_metrics = metrics(fixed_test_path["actual"], fixed_test_path["prediction"])

# Rolling refit test.
selected_test_path, selected_test_refit_count, selected_rolling_params = rolling_refit_forecast(
    train_validation_df,
    test_df,
    FEATURE_COLS,
    SELECTED_TARGET_MODE,
    refit_every=ROLLING_REFIT_EVERY
)
rolling_test_metrics = metrics(selected_test_path["actual"], selected_test_path["prediction"])

print("\nSelected XGBoost fixed test diagnostic")
display(pd.DataFrame([{**fixed_test_metrics, "candidate_id": SELECTED_CANDIDATE_ID, "forecast_mode": "fixed_model_one_step_test"}]))

print("\nSelected XGBoost rolling refit test performance")
display(pd.DataFrame([{**rolling_test_metrics, "candidate_id": SELECTED_CANDIDATE_ID, "forecast_mode": "rolling_refit_one_step_test", "refit_count": selected_test_refit_count}]))

# 5) Forecast path for frontend.
fixed_test_map = fixed_test_path.set_index("date")["prediction"]
forecast_path = selected_test_path.copy()
forecast_path["fixed_model_forecast"] = forecast_path["date"].map(fixed_test_map)
forecast_path["rolling_forecast"] = forecast_path["prediction"]
forecast_path["static_forecast"] = forecast_path["fixed_model_forecast"]
forecast_path["forecast_mode_default"] = "rolling_refit_one_step"

# 6) Feature importance.
importance_records = []
try:
    booster = selected_fixed_model.get_booster()
    score_gain = booster.get_score(importance_type="gain")
    for feature_name, gain in score_gain.items():
        importance_records.append({
            "feature": feature_name,
            "importance_gain": float(gain),
            "target_mode": SELECTED_TARGET_MODE,
        })
    importance_records = sorted(importance_records, key=lambda x: x["importance_gain"], reverse=True)
except Exception as e:
    importance_records = [{"error": str(e)}]

# 7) Optional SHAP summary on selected fixed model.
shap_summary_records = []
if SHAP_AVAILABLE:
    try:
        trainval_clean, X_trainval, y_trainval = clean_xy(train_validation_df, FEATURE_COLS, SELECTED_TARGET_MODE)
        sample_X = X_trainval.sample(n=min(500, len(X_trainval)), random_state=RANDOM_SEED)
        explainer = shap.TreeExplainer(selected_fixed_model)
        shap_values = explainer.shap_values(sample_X)
        mean_abs = np.abs(shap_values).mean(axis=0)
        shap_summary_records = [
            {"feature": feature, "mean_abs_shap": float(value), "target_mode": SELECTED_TARGET_MODE}
            for feature, value in sorted(zip(sample_X.columns, mean_abs), key=lambda x: x[1], reverse=True)
        ]
    except Exception as e:
        shap_summary_records = [{"error": str(e)}]
else:
    shap_summary_records = [{"note": "SHAP package not available in this runtime."}]

# 8) Professor-style charts.
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(forecast_path["date"], forecast_path["actual"], label="Actual Gold Price")
ax.plot(forecast_path["date"], forecast_path["rolling_forecast"], label=f"XGBoost Rolling Refit Forecast ({SELECTED_TARGET_MODE})")
ax.plot(forecast_path["date"], forecast_path["fixed_model_forecast"], label="Fixed Model Diagnostic", alpha=0.65)
ax.set_title("XGBoost — Test Actual vs Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(forecast_path["date"], forecast_path["residual"], label="Rolling Residual = Actual - Forecast")
ax.axhline(0, linewidth=1)
ax.set_title("XGBoost — Rolling Test Residuals")
ax.set_xlabel("Date")
ax.set_ylabel("Residual")
ax.legend()
plt.show()

zoom_df = forecast_path.tail(90)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(zoom_df["date"], zoom_df["actual"], marker="o", linestyle="-", label="Actual Gold Price")
ax.plot(zoom_df["date"], zoom_df["rolling_forecast"], marker="o", linestyle="-", label="XGBoost Rolling Refit Forecast")
ax.set_title("XGBoost — Recent Test Zoom")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
ax.legend()
plt.show()

importance_df = pd.DataFrame(importance_records)
if "importance_gain" in importance_df.columns:
    display(importance_df.head(20))
    fig, ax = plt.subplots(figsize=(12, 6))
    top_imp = importance_df.head(15).sort_values("importance_gain")
    ax.barh(top_imp["feature"], top_imp["importance_gain"])
    ax.set_title("XGBoost Feature Importance — Gain")
    ax.set_xlabel("Gain")
    plt.show()

# Objects used by Cell 5.
xgboost_summary = {
    "notebook": "09_xgboost_forecasting_candidate.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "project_identity": "Gold Nexus Alpha professor-style forecasting platform",
    "dataset": {
        "name": "Dataset B — Core Multivariate",
        "target": TARGET_COL,
        "start": CORE_MULTIVARIATE_START,
        "end": CORE_MULTIVARIATE_END,
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": len(train_df)},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": len(validation_df)},
        "test": {"start": TEST_START, "end": TEST_END, "rows": len(test_df)},
    },
    "excluded_variables": EXCLUDED_FROM_MAIN_MODEL,
    "feature_policy": "Uses gold history features and lagged macro variables; high_yield is excluded from the main model.",
    "gpu_policy": "Notebook attempts Colab GPU/CUDA XGBoost first and falls back to CPU if unavailable.",
    "selection_rule": "Top fixed one-step candidates are rolled through validation; final XGBoost candidate selected by rolling validation RMSE.",
    "selected_model": {
        "candidate_id": SELECTED_CANDIDATE_ID,
        "target_mode": SELECTED_TARGET_MODE,
        "features": FEATURE_COLS,
        "validation_metrics": selected_row,
        "test_metrics_rolling": rolling_test_metrics,
        "test_metrics_fixed_diagnostic": fixed_test_metrics,
        "refit_every": ROLLING_REFIT_EVERY,
        "test_refit_count": int(selected_test_refit_count),
    },
    "fixed_validation_leaderboard": fixed_validation_table.to_dict(orient="records"),
    "rolling_validation_leaderboard": rolling_validation_table.to_dict(orient="records"),
    "interpretation_notes": [
        "Raw price-level XGBoost can underforecast when test prices exceed the training range.",
        "Change, log-price, and log-return targets are tested to reduce extrapolation weakness.",
        "The default frontend chart should use the rolling refit forecast."
    ],
}

xgboost_forecast_path_df = forecast_path.copy()
xgboost_feature_importance_records = importance_records
xgboost_shap_summary_records = shap_summary_records

In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Notebook 09 JSON artifacts for the XGBoost page and validation notebook.
# ======================================================================================

results_path = MODELS_ARTIFACTS_DIR / "xgboost_results.json"
forecast_path_json = MODELS_ARTIFACTS_DIR / "xgboost_forecast_path.json"
feature_importance_path = INTERPRETABILITY_DIR / "xgboost_feature_importance.json"
shap_summary_path = INTERPRETABILITY_DIR / "xgboost_shap_summary.json"
page_path = PAGES_ARTIFACTS_DIR / "page_xgboost.json"

write_json(results_path, xgboost_summary)

write_json(forecast_path_json, {
    "model_name": "XGBoost",
    "selected_candidate_id": xgboost_summary["selected_model"]["candidate_id"],
    "target_mode": xgboost_summary["selected_model"]["target_mode"],
    "forecast_mode_default": "rolling_refit_one_step",
    "fixed_model_forecast_is_diagnostic": True,
    "records": path_records(xgboost_forecast_path_df),
})

write_json(feature_importance_path, {
    "model_name": "XGBoost",
    "selected_candidate_id": xgboost_summary["selected_model"]["candidate_id"],
    "target_mode": xgboost_summary["selected_model"]["target_mode"],
    "importance_type": "gain",
    "records": xgboost_feature_importance_records,
})

write_json(shap_summary_path, {
    "model_name": "XGBoost",
    "selected_candidate_id": xgboost_summary["selected_model"]["candidate_id"],
    "target_mode": xgboost_summary["selected_model"]["target_mode"],
    "records": xgboost_shap_summary_records,
})

page_payload = {
    "page_title": "XGBoost Forecasting Candidate",
    "page_subtitle": "Advanced nonlinear tabular model with target-transform testing and rolling refit evaluation.",
    "artifact_type": "model_page",
    "model_family": "XGBoost",
    "default_chart_mode": "rolling_refit_one_step",
    "dataset_window": xgboost_summary["dataset"],
    "selected_model": xgboost_summary["selected_model"],
    "leaderboards": {
        "fixed_validation": xgboost_summary["fixed_validation_leaderboard"],
        "rolling_validation": xgboost_summary["rolling_validation_leaderboard"],
    },
    "charts": [
        {
            "chart_id": "actual_vs_rolling_forecast",
            "title": "Actual Gold Price vs Rolling XGBoost Forecast",
            "source_artifact": "artifacts/models/xgboost_forecast_path.json",
            "x": "date",
            "y_actual": "actual",
            "y_forecast": "rolling_forecast"
        },
        {
            "chart_id": "fixed_model_diagnostic",
            "title": "Fixed Model Forecast Diagnostic",
            "source_artifact": "artifacts/models/xgboost_forecast_path.json",
            "x": "date",
            "y_actual": "actual",
            "y_forecast": "fixed_model_forecast"
        },
        {
            "chart_id": "feature_importance",
            "title": "XGBoost Feature Importance",
            "source_artifact": "artifacts/interpretability/xgboost_feature_importance.json"
        }
    ],
    "limitations": xgboost_summary["interpretation_notes"],
    "json_first_rule": "Frontend should read all XGBoost claims, metrics, target mode, and selected-model labels from this artifact."
}

write_json(page_path, page_payload)

print("✅ Notebook 09 v2 exports complete")
print(results_path)
print(forecast_path_json)
print(feature_importance_path)
print(shap_summary_path)
print(page_path)

In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 09 revised outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/xgboost_results.json",
    "artifacts/models/xgboost_forecast_path.json",
    "artifacts/interpretability/xgboost_feature_importance.json",
    "artifacts/interpretability/xgboost_shap_summary.json",
    "artifacts/pages/page_xgboost.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Revise Notebook 09 XGBoost rolling artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ GitHub push complete.")
else:
    print("✅ No changes to commit. Artifacts already match repository.")

print("✅ Notebook 09 v2 professor-style XGBoost revision complete.")